# Cox Proportional Hazard Analysis using the Rhino SDK

*Notebook Last Validated: 2026-08-06*

For more info, check out [RhinoDocs](https://docs.rhinofcp.com/rhino-sdk/calculating-federated-analytics-part-2#cox-proportional-hazard)

## What does this notebook do?

This notebook runs a **Cox proportional hazard** analysis across multiple datasets using the Rhino Federated Computing Platform (FCP).

### What is Cox proportional hazard analysis?

Cox regression is a statistical method for answering questions like:

> *"Which factors affect **how long** it takes for a specific event to happen — and by how much?"*

Common examples in healthcare:
- Does a certain treatment reduce the time until a patient recovers?
- Do factors like age or a lab value predict whether — and how soon — a patient experiences a health event?

For each person in your data, the analysis needs two key pieces of information:
- **Time** — how long until the event occurred (or how long the person was observed if it never did)
- **Event** — whether the event actually happened (`1` = yes, `0` = no / still being observed)

It also accounts for **covariates** — other variables that might influence the outcome, such as clinical measurements or demographics.

The result is a set of coefficients (one per covariate) that describe how strongly each factor is associated with the timing of the event.

### What does "federated" mean here?

In a federated analysis, the raw patient data **never leaves the sites that hold it**. Instead, the Rhino FCP coordinates the computation across sites, and only aggregated statistical results are shared. This lets you run analyses on larger, more diverse populations while preserving data privacy.

### What you'll need

- A Rhino FCP account with access to a project
- Two or more datasets in that project, each containing a time column, an event column, and covariate columns
- The `rhino_health` Python package (see Setup below)

## Setup

Ensure you've installed the `rhino_health` SDK before running this notebook. Python 3.8 or later is required.

In [ ]:
%pip install rhino-health

## Step 1: Load Libraries

This cell imports the tools this notebook depends on. You don't need to understand what each one does — just run it before anything else.

In [ ]:
from getpass import getpass
import rhino_health
import pandas as pd
from rhino_health.lib.metrics import *

## Step 2: Log in to the Rhino FCP

Enter your Rhino FCP credentials to start a session. This is the same email and password you use to log in at the Rhino FCP web portal.

When you run the cell below, a password prompt will appear — your password is never stored or displayed.

In [ ]:
my_username = "<ENTER_YOUR_USERNAME>" # Replace this with the email you use to log into Rhino Health

print("Logging In")
session = rhino_health.login(username=my_username, password=getpass())
print("Logged In")

## Step 3: Select Your Project

A **project** in Rhino FCP is the workspace that contains your datasets and defines which sites are participating in the analysis.

Replace `PROJECT_UID` with the exact UID of your project as it appears in the Rhino FCP platform.

In [ ]:
PROJECT_UID = "<ENTER_YOUR_PROJECT_UID>"   # Replace with your Project UID

project   = session.project.get_projects(project_uids=[PROJECT_UID])[0]
workgroup = session.project.get_collaborating_workgroups(PROJECT_UID)[0]

print(f"Project \t({project.uid}): \t{project.name}")
print(f"Workgroup \t({workgroup.uid}): \t{workgroup.name}")

## Step 4: Select Your Datasets

A **dataset** is a table of patient data registered with the platform at a particular site. You'll select the datasets you want to include in this analysis — typically one per participating site.

Replace `DATASET_1` and `DATASET_2` with the exact names of your datasets (should already be registered to the platform). 
Add or remove entries from the list if you have a different number of sites.

In [ ]:
dataset_uids = [
    project.get_dataset_by_name("<ENTER_DATASET_1_NAME>"), # Replace with the name of your first dataset
    project.get_dataset_by_name("<ENTER_DATASET_2_NAME>"), # Replace with the name of your second dataset
]
print(f"Dataset 1 Loaded: {dataset_uids[0].name} ({dataset_uids[0].uid})")
print(f"Dataset 2 Loaded: {dataset_uids[1].name} ({dataset_uids[1].uid})")


# NOTE: if demoing internally (Using the Rhino Health Workgroup), register/use the following dummy datasets:
# - /rhino_data/external/import-external-datasets-dev/DATASET_1.csv
# - /rhino_data/external/import-external-datasets-dev/DATASET_2.csv

### Expected data format

Each dataset must contain at minimum these columns:

| Column | Description | Example values |
|--------|-------------|---------------|
| `Time` | How long (in any consistent unit, e.g. days) until the event occurred, or until the observation ended | `84.0`, `97.0` |
| `Event` | Whether the event happened: `1` = yes, `0` = no (censored — the observation ended before the event occurred) | `1`, `0` |
| Covariate columns | Any additional variables you want to test as potential influencing factors | `0.3`, `5.3` |

The cell below shows what a small example dataset looks like. Your real datasets will have many more rows.

In [ ]:
pd.DataFrame({
    'Time': [84.0, 97.0, 91.0, 90.0, 124.0, 97.0],
    'Event': [1, 0, 0, 1, 1, 1],
    'COV1': [0.3, 0.51, 0.12, 0.03, 0.413, 0.3],
    'COV2': [5.3, 1.51, 1.8, 0.03, 13, 0.3]
})

## Step 5: Configure and Run the Analysis

Now we tell the platform which columns in your data correspond to the time, event, and covariates, then kick off the federated computation.

**Update the three variables below to match your dataset's column names:**

- `time_variable` — the name of the column that records how long until the event (or end of observation)
- `event_variable` — the name of the column that records whether the event occurred (`1`/`0`)
- `covariates` — a list of column names for the factors you want to test

The analysis uses an iterative process (up to `max_iterations=50` rounds) to converge on the best-fit coefficients. The platform coordinates this across all participating sites automatically — no manual steps are needed between rounds.

Once complete, the `results` object will contain the model coefficients, which describe the relationship between each covariate and the timing of the event.

In [ ]:
# Set the time and event variables
time_variable = "Time"
event_variable = "Event"
covariates = ["COV1", "COV2"]

# Create a Cox instance, use the mean of the local betas of the two sites as the initial beta
metric_configuration = Cox(time_variable=time_variable, event_variable=event_variable, covariates=covariates, initial_beta="mean", max_iterations=50)

# Retrieve results for your project and datasets
results = project.aggregate_dataset_metric(dataset_uids=[str(dataset.uid) for dataset in dataset_uids], metric_configuration=metric_configuration)

print(f"Analysis Complete. Proceed to Step 6 to view results.")

## Step 6: View the Results

The `results` object returned above contains the full Cox model output. Run the cells below to display and interpret it.

> **Note:** Results are returned directly to this notebook — they do not appear in the Rhino FCP web UI. The metrics API is a lightweight, synchronous call; only containerized code runs (NVFlare, generalized compute) create entries in the UI.

In [ ]:
import math
import numpy as np

# The raw output is keyed by "cox", with lists ordered to match `covariates`
cox_out  = results.output["cox"]
betas    = cox_out["beta_vector"]   # log hazard ratios, one per covariate
std_errs = cox_out["std_error"]     # standard errors

# Derive the statistics that are useful for interpretation
z_scores = [b / se for b, se in zip(betas, std_errs)]
p_values = [math.erfc(abs(z) / math.sqrt(2)) for z in z_scores]   # two-tailed
hrs      = [math.exp(b) for b in betas]
hr_lo    = [math.exp(b - 1.96 * se) for b, se in zip(betas, std_errs)]
hr_hi    = [math.exp(b + 1.96 * se) for b, se in zip(betas, std_errs)]

summary = pd.DataFrame({
    "Coefficient":     betas,
    "Hazard Ratio":    hrs,
    "Std Error":       std_errs,
    "Z-score":         z_scores,
    "P-value":         p_values,
    "HR 95% CI Lower": hr_lo,
    "HR 95% CI Upper": hr_hi,
}, index=pd.Index(covariates, name="Covariate"))

summary.round(4)

In [ ]:
# Quick significance summary
print("Significance check (p < 0.05 threshold):\n")
for cov, p, hr, lo, hi in zip(covariates, p_values, hrs, hr_lo, hr_hi):
    sig = "SIGNIFICANT" if p < 0.05 else "not significant"
    direction = "increases" if hr > 1 else "decreases"
    print(f"  {cov}: p={p:.3f} → {sig}")
    if p < 0.05:
        print(f"    Higher values {direction} event risk (HR={hr:.3f}, 95% CI: {lo:.3f}–{hi:.3f})")

### How to interpret the results

The SDK returns two raw values per covariate — `beta_vector` and `std_error`. The code above uses those to derive the statistics that are actually meaningful:

| Column | What it is | How to read it |
|--------|-----------|---------------|
| **Coefficient** | The raw beta from the model (log hazard ratio) | Sign matters more than magnitude: negative = the event tends to happen later as this variable increases; positive = sooner. Not intuitive on its own — use Hazard Ratio instead. |
| **Hazard Ratio** | `exp(Coefficient)` — the most practical number | `1.0` = no effect. `2.0` = the event is twice as likely to occur at any moment for a one-unit increase in this variable. `0.5` = half as likely (protective). |
| **Std Error** | Uncertainty in the coefficient estimate | Smaller = more precise. Large std errors relative to the coefficient suggest the model isn't confident. |
| **Z-score** | Coefficient ÷ Std Error | How many standard errors the coefficient is away from zero. Rule of thumb: absolute value of z > 1.96 corresponds to p < 0.05. |
| **P-value** | Probability of seeing this result by chance if there were truly no effect | Below `0.05` is a common threshold for "statistically significant." Above it, the data don't provide strong evidence of a real association. |
| **HR 95% CI Lower/Upper** | Range the true hazard ratio likely falls within | If this range includes `1.0`, you can't confidently say the variable has an effect in either direction. |

#### An Example Interpretation (Using Demo Data)

- **COV1**: Coefficient ≈ −1.33, Hazard Ratio ≈ 0.26. This suggests higher COV1 is associated with the event happening later (HR < 1), but with a large std error (0.95) and p ≈ 0.16, there isn't enough statistical evidence to be confident.
- **COV2**: Coefficient ≈ −0.03, Hazard Ratio ≈ 0.97. Nearly no effect detected, and p ≈ 0.45 — the data show no meaningful association.

> **Why aren't the results significant?** The example datasets used have only 6 rows each. Cox regression needs much larger samples to detect real effects reliably. With real clinical datasets across sites, you should see more stable and meaningful results.

## Step 7: Visualize the Results

### Forest plot
The standard way to visualize Cox results is a **forest plot** — one row per covariate, showing the hazard ratio (dot) and its 95% confidence interval (horizontal line). The vertical dashed line at HR = 1 is the "no effect" reference: CIs that cross it are not statistically conclusive.

Statistically significant covariates (p < 0.05) are shown in blue; non-significant ones are shown in gray.

In [ ]:
%pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Palette
_SIG    = "#2a78d6"   # blue — significant (p < 0.05)
_MUTED  = "#898781"   # gray — non-significant
_REF    = "#c3c2b7"   # reference line / axis
_INK    = "#0b0b0b"   # primary text
_INK2   = "#52514e"   # secondary text
_GRID   = "#e1e0d9"   # recessive gridlines
_BG     = "#fcfcfb"   # chart surface

n = len(covariates)
y_pos = list(range(n - 1, -1, -1))   # top → bottom

fig, ax = plt.subplots(figsize=(9, max(3.0, n * 1.1 + 2.0)))
fig.patch.set_facecolor(_BG)
ax.set_facecolor(_BG)

x_label_start = max(hr_hi) * 1.4

for y, cov, hr, lo, hi, p in zip(y_pos, covariates, hrs, hr_lo, hr_hi, p_values):
    color = _SIG if p < 0.05 else _MUTED
    ax.hlines(y, lo, hi, color=color, linewidth=2, zorder=3)
    ax.scatter(hr, y, color=color, s=72, zorder=5, linewidths=2, edgecolors=_BG)
    ax.text(x_label_start, y,
            f"HR {hr:.2f}  [{lo:.2f}–{hi:.2f}]  p={p:.2f}",
            va="center", ha="left", fontsize=8.5, color=_INK2)

ax.axvline(x=1, color=_REF, linewidth=1, zorder=2)
ax.text(1, n - 0.45, "No effect", ha="center", va="bottom",
        fontsize=7.5, color=_INK2, style="italic")

ax.set_xscale("log")
ax.set_xlim(left=min(hr_lo) * 0.6, right=max(hr_hi) * 2.2)
ax.set_yticks(y_pos)
ax.set_yticklabels(covariates, fontsize=9, color=_INK)
ax.set_xlabel("Hazard Ratio  (log scale, 95% CI)", fontsize=9, color=_INK)
ax.set_title("Cox Proportional Hazard — Forest Plot",
             fontsize=11, color=_INK, pad=12, fontweight="semibold")

ax.xaxis.grid(True, color=_GRID, linewidth=1, zorder=1)
ax.set_axisbelow(True)
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(_REF)
ax.tick_params(colors=_INK, labelsize=8.5)

# Legend anchored to the figure (not the axes) so it never overlaps axis labels
sig_patch   = mpatches.Patch(color=_SIG,   label="p < 0.05  (significant)")
muted_patch = mpatches.Patch(color=_MUTED, label="p ≥ 0.05  (not significant)")
fig.legend(handles=[sig_patch, muted_patch], fontsize=8, frameon=False,
           ncol=2, labelcolor=_INK, loc="lower center")

# Reserve 10% at the bottom so tight_layout doesn't let the axes crowd the legend
plt.tight_layout(rect=[0, 0.10, 1, 1])
plt.show()

---

## Step 8: Kaplan-Meier Survival Curve

A **Kaplan-Meier curve** shows survival probability over time — the probability that the event has *not yet* occurred by a given point. It complements Cox regression: Cox tells you *which factors* matter and by how much; KM shows you *when* events happen across the whole population.

The `KaplanMeier` metric is a separate API call from `Cox`. It returns the raw time and event vectors, which you then feed into a survival model to get the curve.

> **Important:** the output contains raw data (times and 0/1 event flags), not pre-computed probabilities. Plotting it directly produces a meaningless scatter of 0s and 1s. Always use `surv_func_right_model()` to compute the actual survival curve first.

In [ ]:
%pip install statsmodels

In [ ]:
from rhino_health.lib.metrics import KaplanMeier

km_config = KaplanMeier(time_variable=time_variable, event_variable=event_variable)
km_results = project.aggregate_dataset_metric(
    dataset_uids=[str(d.uid) for d in dataset_uids],
    metric_configuration=km_config
)

# Compute the survival function — do not plot the raw output directly
surv_model = km_results.surv_func_right_model()
print(surv_model.summary())

In [ ]:
import matplotlib.pyplot as plt

_INK    = "#0b0b0b"
_INK2   = "#52514e"
_SIG    = "#2a78d6"
_GRID   = "#e1e0d9"
_BG     = "#fcfcfb"

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(_BG)
ax.set_facecolor(_BG)

# Survival curve: step down at each event time
ax.step(surv_model.surv_times, surv_model.surv_prob,
        where="post", color=_SIG, linewidth=2, label="Survival estimate")

# Shade the 95% confidence band
lower, upper = surv_model.simultaneous_cb()
ax.fill_between(surv_model.surv_times, lower, upper,
                step="post", alpha=0.15, color=_SIG, label="95% CI")

# Cosmetics
ax.set_ylim(0, 1.05)
ax.set_xlabel(f"Time  ({time_variable})", fontsize=9, color=_INK)
ax.set_ylabel("Survival probability  S(t)", fontsize=9, color=_INK)
ax.set_title("Kaplan-Meier Survival Curve", fontsize=11, color=_INK,
             pad=12, fontweight="semibold")
ax.axhline(0.5, color=_INK2, linewidth=1, linestyle="dashed", alpha=0.5)
ax.text(ax.get_xlim()[1], 0.5, " median", va="center",
        fontsize=7.5, color=_INK2, style="italic")

ax.xaxis.grid(True, color=_GRID, linewidth=1)
ax.yaxis.grid(True, color=_GRID, linewidth=1)
ax.set_axisbelow(True)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["bottom", "left"]:
    ax.spines[spine].set_color(_GRID)
ax.tick_params(colors=_INK, labelsize=8.5)
ax.legend(fontsize=8, frameon=False, labelcolor=_INK)

plt.tight_layout()
plt.show()

### How to read a Kaplan-Meier curve

**Y-axis — survival probability S(t):** starts at 1.0 (100% of patients event-free at time zero) and decreases toward 0 as events accumulate. A value of 0.6 at time 90 means "60% of patients had not yet experienced the event by day 90."

**X-axis — time:** whatever unit your `Time` column uses (days, months, etc.).

**Each step down** marks when one or more events occurred in the combined dataset. The curve only moves at event times — it stays flat between events.

**The shaded band** is the 95% confidence interval. Wide bands mean high uncertainty — usually because there are few patients in the dataset. A narrow band means the estimate is reliable.

**The dashed line at 0.5** marks median survival — the point at which half the population has experienced the event. If the curve never crosses 0.5, median survival was not reached within the observation period.

---

#### Why does the curve look unusual with the demo data?

With only 6 patients across two sites, each event causes a drop of roughly 17–25% in one step. Real clinical KM curves with hundreds or thousands of patients look smoother because events are spread across more time points.